# XGBoost 辅助模型调参

`06_submission_weighted_blend.csv` 的公榜分数为 `0.12549`。当前最有效的方向是 Lasso + XGBoost 融合。本 notebook 固定已验证有效的数据处理方式，只调 XGBoost 参数，并重新搜索 Lasso / XGBoost 融合权重。

In [1]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LassoCV
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBRegressor

warnings.filterwarnings("ignore", category=UserWarning)
plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "Arial Unicode MS"]
plt.rcParams["axes.unicode_minus"] = False

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = ROOT / "data"
SUBMISSION_DIR = ROOT / "submissions"

train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
train.shape, test.shape

((1460, 81), (1459, 80))

## 1. 复用当前最佳数据处理

In [2]:
target = "SalePrice"
id_column = "Id"

outlier_mask = (train["GrLivArea"] > 4000) & (train["SalePrice"] < 300000)
train_model = train.loc[~outlier_mask].copy()

X = train_model.drop(columns=[target, id_column])
y = train_model[target]
X_test = test.drop(columns=[id_column]).copy()
test_ids = test[id_column]

print("移除异常点数量:", int(outlier_mask.sum()))
print("异常点 Id:", train.loc[outlier_mask, id_column].tolist())

移除异常点数量: 2
异常点 Id: [524, 1299]


In [3]:
def log_transform_skewed_features(X_train: pd.DataFrame, X_test: pd.DataFrame, threshold: float = 0.75):
    X_train = X_train.copy()
    X_test = X_test.copy()
    numeric_columns = X_train.select_dtypes(include="number").columns
    skewness = X_train[numeric_columns].skew().sort_values(ascending=False)
    skewed_columns = skewness[skewness > threshold].index.tolist()

    transformed_columns = []
    for column in skewed_columns:
        min_value = min(X_train[column].min(), X_test[column].min())
        if pd.notna(min_value) and min_value >= 0:
            X_train[column] = np.log1p(X_train[column])
            X_test[column] = np.log1p(X_test[column])
            transformed_columns.append(column)

    return X_train, X_test, transformed_columns


X_log, X_test_log, transformed_columns = log_transform_skewed_features(X, X_test)
print("log1p 处理的列数量:", len(transformed_columns))

log1p 处理的列数量: 20


## 2. 建模函数

In [4]:
def make_one_hot_encoder() -> OneHotEncoder:
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def build_model(base_model, scale_numeric: bool = False) -> TransformedTargetRegressor:
    numeric_features = X_log.select_dtypes(include="number").columns.tolist()
    categorical_features = X_log.select_dtypes(exclude="number").columns.tolist()

    numeric_steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale_numeric:
        numeric_steps.append(("scaler", StandardScaler()))

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", Pipeline(numeric_steps), numeric_features),
            (
                "cat",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="most_frequent")),
                        ("onehot", make_one_hot_encoder()),
                    ]
                ),
                categorical_features,
            ),
        ]
    )

    return TransformedTargetRegressor(
        regressor=Pipeline(steps=[("preprocessor", preprocessor), ("model", base_model)]),
        func=np.log1p,
        inverse_func=np.expm1,
    )


def rmsle(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    y_pred = np.maximum(y_pred, 0)
    return float(np.sqrt(mean_squared_error(np.log1p(y_true), np.log1p(y_pred))))

## 3. 参数候选

调参目标不是让 XGBoost 单模型最强，而是让它作为 Lasso 的互补模型更稳定。第一轮发现 `depth=2` 偏弱，`depth=3` 配合更强正则更适合融合。

In [5]:
xgb_configs = {
    "current": dict(
        n_estimators=900,
        learning_rate=0.03,
        max_depth=3,
        subsample=0.85,
        colsample_bytree=0.75,
        reg_lambda=1.5,
        reg_alpha=0.001,
    ),
    "depth3_lambda7": dict(
        n_estimators=1100,
        learning_rate=0.025,
        max_depth=3,
        subsample=0.80,
        colsample_bytree=0.70,
        reg_lambda=7,
        reg_alpha=0.01,
    ),
    "depth3_lambda10": dict(
        n_estimators=1100,
        learning_rate=0.025,
        max_depth=3,
        subsample=0.80,
        colsample_bytree=0.70,
        reg_lambda=10,
        reg_alpha=0.01,
    ),
    "depth3_lambda3_sub85": dict(
        n_estimators=1100,
        learning_rate=0.025,
        max_depth=3,
        subsample=0.85,
        colsample_bytree=0.70,
        reg_lambda=3,
        reg_alpha=0.01,
    ),
    "depth2_lambda5": dict(
        n_estimators=1400,
        learning_rate=0.02,
        max_depth=2,
        subsample=0.80,
        colsample_bytree=0.70,
        reg_lambda=5,
        reg_alpha=0.01,
    ),
}
xgb_configs

{'current': {'n_estimators': 900,
  'learning_rate': 0.03,
  'max_depth': 3,
  'subsample': 0.85,
  'colsample_bytree': 0.75,
  'reg_lambda': 1.5,
  'reg_alpha': 0.001},
 'depth3_lambda7': {'n_estimators': 1100,
  'learning_rate': 0.025,
  'max_depth': 3,
  'subsample': 0.8,
  'colsample_bytree': 0.7,
  'reg_lambda': 7,
  'reg_alpha': 0.01},
 'depth3_lambda10': {'n_estimators': 1100,
  'learning_rate': 0.025,
  'max_depth': 3,
  'subsample': 0.8,
  'colsample_bytree': 0.7,
  'reg_lambda': 10,
  'reg_alpha': 0.01},
 'depth3_lambda3_sub85': {'n_estimators': 1100,
  'learning_rate': 0.025,
  'max_depth': 3,
  'subsample': 0.85,
  'colsample_bytree': 0.7,
  'reg_lambda': 3,
  'reg_alpha': 0.01},
 'depth2_lambda5': {'n_estimators': 1400,
  'learning_rate': 0.02,
  'max_depth': 2,
  'subsample': 0.8,
  'colsample_bytree': 0.7,
  'reg_lambda': 5,
  'reg_alpha': 0.01}}

## 4. OOF 调参评估

In [6]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)

lasso_model = build_model(
    LassoCV(alphas=np.logspace(-4, 0, 40), cv=5, max_iter=30000, random_state=42),
    scale_numeric=True,
)

lasso_oof = np.zeros(len(X_log))
for train_idx, valid_idx in cv.split(X_log):
    fold_model = clone(lasso_model)
    fold_model.fit(X_log.iloc[train_idx], y.iloc[train_idx])
    lasso_oof[valid_idx] = fold_model.predict(X_log.iloc[valid_idx])

rows = []
for name, params in xgb_configs.items():
    xgb_model = build_model(
        XGBRegressor(objective="reg:squarederror", random_state=42, n_jobs=-1, **params),
        scale_numeric=False,
    )
    xgb_oof = np.zeros(len(X_log))
    for train_idx, valid_idx in cv.split(X_log):
        fold_model = clone(xgb_model)
        fold_model.fit(X_log.iloc[train_idx], y.iloc[train_idx])
        xgb_oof[valid_idx] = fold_model.predict(X_log.iloc[valid_idx])

    best_weight = None
    best_score = 999
    for w_lasso in np.arange(0.60, 0.701, 0.01):
        prediction = w_lasso * lasso_oof + (1 - w_lasso) * xgb_oof
        score = rmsle(y, prediction)
        if score < best_score:
            best_weight = w_lasso
            best_score = score

    rows.append(
        {
            "方案": name,
            "xgb_oof": rmsle(y, xgb_oof),
            "w_lasso": round(best_weight, 2),
            "w_xgboost": round(1 - best_weight, 2),
            "blend_oof": best_score,
            **params,
        }
    )

tuning_results = pd.DataFrame(rows).sort_values("blend_oof")
tuning_results

,方案,xgb_oof,w_lasso,w_xgboost,blend_oof,n_estimators,learning_rate,max_depth,subsample,colsample_bytree,reg_lambda,reg_alpha
1,depth3_lambda7,0.114901,0.63,0.37,0.107594,1100,0.025,3,0.80,0.70,7.0,0.010
3,depth3_lambda3_sub85,0.115461,0.64,0.36,0.107632,1100,0.025,3,0.85,0.70,3.0,0.010
2,depth3_lambda10,0.115168,0.64,0.36,0.107743,1100,0.025,3,0.80,0.70,10.0,0.010
0,current,0.115779,0.65,0.35,0.107800,900,0.030,3,0.85,0.75,1.5,0.001
4,depth2_lambda5,0.116060,0.68,0.32,0.108367,1400,0.020,2,0.80,0.70,5.0,0.010


## 5. 生成调参后提交文件

In [7]:
best = tuning_results.iloc[0]
best_params = xgb_configs[best["方案"]]
w_lasso = float(best["w_lasso"])
w_xgboost = float(best["w_xgboost"])

final_lasso = clone(lasso_model)
final_xgboost = build_model(
    XGBRegressor(objective="reg:squarederror", random_state=42, n_jobs=-1, **best_params),
    scale_numeric=False,
)

final_lasso.fit(X_log, y)
final_xgboost.fit(X_log, y)

final_prediction = np.maximum(
    w_lasso * final_lasso.predict(X_test_log) + w_xgboost * final_xgboost.predict(X_test_log),
    0,
)

submission = pd.DataFrame({"Id": test_ids, "SalePrice": final_prediction})
SUBMISSION_DIR.mkdir(exist_ok=True)
submission_path = SUBMISSION_DIR / "07_submission_xgboost_tuned_blend.csv"
submission.to_csv(submission_path, index=False)

best["方案"], w_lasso, w_xgboost, submission_path, submission.head(), submission["SalePrice"].describe()

('depth3_lambda7',
 0.63,
 0.37,
 WindowsPath('C:/Users/84740/house-prices-advanced-regression/submissions/07_submission_xgboost_tuned_blend.csv'),
      Id      SalePrice
 0  1461  120984.059166
 1  1462  153899.071413
 2  1463  183783.485114
 3  1464  196585.732435
 4  1465  193484.108709,
 count      1459.000000
 mean     178731.823526
 std       78781.471510
 min       42392.578800
 25%      126455.976127
 50%      157793.195583
 75%      209667.413750
 max      860944.454631
 Name: SalePrice, dtype: float64)

## 6. 提交命令

```powershell
kaggle competitions submit -c house-prices-advanced-regression-techniques -f submissions/07_submission_xgboost_tuned_blend.csv -m "tuned xgboost lasso blend"
```